In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def lattice_spacing_su2(beta: float) -> float:
    """SU(2) lattice spacing from arXiv:1811.02800."""
    t0_phys = 0.01133  # fm^2
    delta_beta = beta - 2.600
    ln_t0_over_a2 = 1.285 + 6.409 * delta_beta - 0.7411 * delta_beta**2
    t0_over_a2 = np.exp(ln_t0_over_a2)
    return np.sqrt(t0_phys / t0_over_a2)


def lattice_spacing_su3(beta: float) -> float:
    """SU(3) lattice spacing from arXiv:hep-lat/0108008."""
    r0 = 0.5  # fm
    delta_beta = beta - 6.0
    ln_a_over_r0 = -1.6804 - 1.7331 * delta_beta + 0.7849 * delta_beta**2 - 0.4428 * delta_beta**3
    return np.exp(ln_a_over_r0) * r0


In [3]:
# ---------------------------------------------------------------------------
# Volume-preserving lattice adjustment
# ---------------------------------------------------------------------------

def adjust_lattice(T_ref: int, L_ref: int, a_ref_fm: float, a_new_fm: float,
                   open_bc: bool = False, n_exclude_ref: int = 2) -> dict:
    """Compute (T_new, L_new) keeping physical volume constant.

    For open BC, n_exclude scales with lattice spacing to keep the
    physical exclusion distance d_phys = n_exclude_ref * a_ref constant:
        n_exclude_new = round(n_exclude_ref * a_ref / a_new)
    T_new is then adjusted so that T_eff * a stays constant.
    """
    ratio = a_ref_fm / a_new_fm
    L_new = round(L_ref * ratio)

    if open_bc:
        # Physical exclusion distance fixed from reference
        d_phys = n_exclude_ref * a_ref_fm
        n_exclude_new = max(1, round(d_phys / a_new_fm))

        T_eff_ref = T_ref - 2 * n_exclude_ref
        T_new = round(T_eff_ref * ratio) + 2 * n_exclude_new
        T_eff_new = T_new - 2 * n_exclude_new
    else:
        n_exclude_new = 0
        T_new = round(T_ref * ratio)
        T_eff_new = T_new

    V_fm4 = T_eff_new * L_new**3 * a_new_fm**4
    return {"T_new": T_new, "L_new": L_new, "T_eff_new": T_eff_new,
            "n_exclude": n_exclude_new, "V_fm4": V_fm4}

In [12]:
# ---------------------------------------------------------------------------
# Scan: beta=2.5 upward, reference T=17, L=13, open BC  [SU(2)]
# ---------------------------------------------------------------------------

T_ref, L_ref = 22, 15
beta_ref  = 2.5
a_ref_fm  = lattice_spacing_su2(beta_ref)
open_bc   = True
n_exclude_ref = 3

beta_values = np.arange(2.5, 3.35, 0.05)

d_phys = n_exclude_ref * a_ref_fm
print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print(f"Physical exclusion distance: {d_phys:.4f} fm  (n_exclude_ref={n_exclude_ref})")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'n_excl':>7} {'V [fm^4]':>12}")
print("-" * 65)

for beta in beta_values:
    a = lattice_spacing_su2(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude_ref=n_exclude_ref)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['n_exclude']:7d} {adj['V_fm4']:12.6f}")

Reference: beta=2.5, T=22, L=15, a=0.0774 fm, open_bc=True
Physical exclusion distance: 0.2323 fm  (n_exclude_ref=3)

    beta     a [fm]   T_new   L_new   T_eff  n_excl     V [fm^4]
-----------------------------------------------------------------
    2.50     0.0774      22      15      16       3     1.940138
    2.55     0.0658      27      18      19       4     2.074162
    2.60     0.0560      30      21      22       4     2.001721
    2.65     0.0477      36      24      26       5     1.867239
    2.70     0.0408      42      28      30       6     1.822539
    2.75     0.0349      49      33      35       7     1.868092
    2.80     0.0299      57      39      41       8     1.952953
    2.85     0.0257      66      45      48       9     1.913097
    2.90     0.0221      76      52      56      10     1.889814
    2.95     0.0191      89      61      65      12     1.957515
    3.00     0.0165     103      70      75      14     1.900892
    3.05     0.0143     119      81 

In [5]:
# ---------------------------------------------------------------------------
# Scan: beta=6.1 upward, reference T=18, L=14, open BC  [SU(3)]
# ---------------------------------------------------------------------------

T_ref, L_ref = 18, 14
beta_ref  = 6.1
a_ref_fm  = lattice_spacing_su3(beta_ref)
open_bc   = True
n_exclude_ref = 2

beta_values = np.arange(6.1, 7.45, 0.05)

d_phys = n_exclude_ref * a_ref_fm
print(f"Reference: beta={beta_ref}, T={T_ref}, L={L_ref}, a={a_ref_fm:.4f} fm, open_bc={open_bc}")
print(f"Physical exclusion distance: {d_phys:.4f} fm  (n_exclude_ref={n_exclude_ref})")
print()
print(f"{'beta':>8} {'a [fm]':>10} {'T_new':>7} {'L_new':>7} {'T_eff':>7} {'n_excl':>7} {'V [fm^4]':>12}")
print("-" * 65)

for beta in beta_values:
    a = lattice_spacing_su3(beta)
    adj = adjust_lattice(T_ref, L_ref, a_ref_fm, a, open_bc=open_bc, n_exclude_ref=n_exclude_ref)
    print(f"{beta:8.2f} {a:10.4f} {adj['T_new']:7d} {adj['L_new']:7d} "
          f"{adj['T_eff_new']:7d} {adj['n_exclude']:7d} {adj['V_fm4']:12.6f}")

Reference: beta=6.1, T=18, L=14, a=0.0789 fm, open_bc=True
Physical exclusion distance: 0.1578 fm  (n_exclude_ref=2)

    beta     a [fm]   T_new   L_new   T_eff  n_excl     V [fm^4]
-----------------------------------------------------------------
    6.10     0.0789      18      14      14       2     1.489478
    6.15     0.0730      19      15      15       2     1.437370
    6.20     0.0677      20      16      16       2     1.378644
    6.25     0.0630      24      18      18       3     1.653259
    6.30     0.0587      25      19      19       3     1.550518
    6.35     0.0549      26      20      20       3     1.449272
    6.40     0.0513      28      22      22       3     1.625800
    6.45     0.0481      29      23      23       3     1.495673
    6.50     0.0451      33      25      25       4     1.613837
    6.55     0.0423      34      26      26       4     1.462903
    6.60     0.0397      36      28      28       4     1.526330
    6.65     0.0373      38      30 

In [6]:
# ---------------------------------------------------------------------------
# Corrected: Expected spread for a FIXED lattice size
# ---------------------------------------------------------------------------

HBAR_C = 197.3  # MeV·fm

# Target susceptibility scales
chi_scales = {
    "SU(2)": 200.0, # MeV
    "SU(3)": 191.0  # MeV
}

# Reference geometries
geometries = {
    "SU(2)": {"T": 20, "L": 16, "beta": 2.5, "func": lattice_spacing_su2},
    "SU(3)": {"T": 20, "L": 16, "beta": 6.1, "func": lattice_spacing_su3}
}

for label in ["SU(2)", "SU(3)"]:
    # 1. Get parameters for this group
    geom = geometries[label]
    a_fm = geom["func"](geom["beta"])
    chi_mev = chi_scales[label]
    
    # 2. Calculate Physical Volume: V = T * L^3 * a^4
    V_phys = (geom["T"] * a_fm) * (geom["L"] * a_fm)**3
    
    # 3. Convert Chi to fm^-4: chi_fm = (chi_mev / 197.3)^4
    chi_fm4 = (chi_mev / HBAR_C)**4
    
    # 4. Calculate Expected <Q^2> and Spread
    expected_q2 = chi_fm4 * V_phys
    spread = np.sqrt(expected_q2)
    
    print(f"--- {label} (beta={geom['beta']}) ---")
    print(f"Lattice: {geom['T']} x {geom['L']}^3, a = {a_fm:.4f} fm")
    print(f"Physical Volume: {V_phys:.4f} fm^4")
    print(f"Expected <Q^2>:  {expected_q2:.2f}")
    print(f"Expected Spread: {spread:.2f} (Standard deviation of Q)")
    print()

--- SU(2) (beta=2.5) ---
Lattice: 20 x 16^3, a = 0.0774 fm
Physical Volume: 2.9433 fm^4
Expected <Q^2>:  3.11
Expected Spread: 1.76 (Standard deviation of Q)

--- SU(3) (beta=6.1) ---
Lattice: 20 x 16^3, a = 0.0789 fm
Physical Volume: 3.1762 fm^4
Expected <Q^2>:  2.79
Expected Spread: 1.67 (Standard deviation of Q)



In [7]:
beta = 3.0
N = 1

a_fm   = lattice_spacing_su2(beta)
X_ref  = X_ref_su2
HBAR_C = 200

chi_t_fm4 = (X_ref / HBAR_C)**4
V         = (N * a_fm)**4

spread = np.sqrt(chi_t_fm4 * V)
print(f"a = {a_fm:.4f} fm,  V = {V:.4f} fm⁴,  spread = {spread:.2f}")


NameError: name 'X_ref_su2' is not defined

In [ ]:
(0.0775 * (16))**8

5.589506702973337

In [ ]:
0.5**2

0.25